# Demo — one agent, one deploy target

One flag: where it runs (`slurm` / `kaggle`). The notebook builds a TAAF benchmark around the agent's own
preset, submits it, and stops.

## Tufa Labs' ARC3 solution

Run top to bottom with `TARGET = "kaggle"`, this notebook pushes the Kaggle notebook `tufa-labs-arc3-solution`,
with its source in the dataset `<owner>/tufa-labs-arc3-solution-src`, both private. They are built the way the
final submission was (scriptVersionId 365702303, public score 70.92), and nothing here touches that
submission. The competition submission itself is made from a finished version in Kaggle's UI:
`run_as_submission` stays off, so nothing here submits.

The final submission's settings, as the version that was submitted pushed them:

- the seal at its `kaggle-rtx` presets, unchanged: joint205-u800 with its PLE shards from our ARC2
  REAP-307 model, the arc23 INT4 drafter, INT4 KV cache, the Flash-Next throughput stack;
- no startup stealth and no startup deadlines, and the keepalive on;

The version run itself plays nine hours. A submission ignores that budget and runs to the competition's
own limit. The one difference that reaches a submission is the vLLM pin: the fork's `tufa/v0.29.0` at #16's merge
(`082824c5`) rather than #16's branch (`5ca56119`). The three commits between them reflow comments, let
a sampler module import on CPU, and fix a test stub, so the engine runs the same code on the GPU.

## This is a demo, not "the" way to run a solver

This repo is a **library**. What it offers is the agent presets
(`SealAgent.build()`, one call per engine/environment coordinate),
`platypus.LLMAgentSolver`, and TAAF's `Benchmark` together with its
deploy targets. `requirements.md` §1 puts *all* configuration on the
agent for exactly this reason: one fully-configured agent instance
describes a whole experiment, so there is nothing left for a launcher to
own and no per-agent solver factory to go looking for.

Nothing in the library imports this notebook, and no test depends on it.
It is one wiring of the public pieces, kept short enough to read in a
sitting. Copy it, edit it in place, or drive the same three objects from
a script — an experiment is an agent instance plus a benchmark plus a
target, and any host that can build those is as good as this one.

What it deliberately does not do: no A/B arms, no multi-pass scoring, no
result analysis, and no reattachment to a running job. Those belong
wherever the experiment lives, not in the library.

## The run is capped at 30 minutes and will not finish

Both targets get a 30-minute budget, so **the score this produces is not
a result**. The budget runs from worker start, so loading a 27B FP8
engine comes out of it before a single move is played. Games still in
flight at the deadline are cancelled and land as `gave_up`; games never
started score nothing at all.

The two targets spend that 30 minutes slightly differently, and the
difference is in TAAF, not here:

- **SLURM** stops at `worker start + max_runtime_s` exactly — the
  10-minute teardown buffer applies only to the *walltime* deadline, so
  it does not come out of the budget. Roughly 30 minutes of work, with
  the rest of the hour's walltime left for teardown.
- **Kaggle** stops at `budget − min(10 min, budget/2)`, because there the
  budget is the kernel's whole runtime and results still have to be
  written inside it. Roughly 20 minutes of work.

That is the point: this demonstrates that an agent, a benchmark and a
target wire together and a job comes back, and nothing more. Anything
you would put in a table needs a real budget and several passes — the
`n_passes=1` here cannot separate an agent from its variance either.

In [ ]:
# --- the flag -------------------------------------------------------------
TARGET = "kaggle"  # "slurm" | "kaggle"

# Demo budget, both targets. TAAF's own soft-deadline machinery enforces
# it (TufaSlurmTarget.max_runtime_s / KaggleTarget.max_runtime_s), stopping
# the run one buffer short of it. On Kaggle it bounds only the version run,
# nine hours there (2026-11-05); a submission ignores it and runs to
# the competition's own limit.
DEMO_BUDGET_S = 9 * 3600

import datetime
import getpass
import pathlib
import tempfile

import taaf.benchmark
import taaf.deploy_kaggle
import taaf.deploy_slurm
import taaf.standard_benchmarks

import platypus.deployment
from platypus import LLMAgentSolver

assert TARGET in ("slurm", "kaggle"), TARGET

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# SLURM writes its artifacts here from the compute node, so the job dir has
# to be on shared storage. A Kaggle deploy only stages a bundle for upload
# from this machine, and nothing reads it afterwards, so it goes to local
# temp — several hundred small files at NFS's ~100 ms per create is minutes
# of deploy time bought for nothing.
RESULTS_ROOT = (
    pathlib.Path("/data/shared/research/arc_3_results")
    if TARGET == "slurm"
    else pathlib.Path(tempfile.gettempdir())
)
JOB_DIR = RESULTS_ROOT / getpass.getuser() / "platypus_demo" / f"seal_{STAMP}"

print(f"target={TARGET} budget={DEMO_BUDGET_S / 60:.0f} min")
print(f"job_dir={JOB_DIR}")

## What the flag selects

The seal plays, configured entirely by its own presets — this notebook
overrides nothing.

**`TARGET`** picks two things at once, because they are not independent:

- *The column.* `build()` is the local/SLURM stack; `build(environment="kaggle-rtx")`
  is the same agent on Kaggle's offline stack — weights from a mounted
  dataset snapshot and vLLM installed `--no-index` from a wheels kernel,
  then replaced with this repo's pinned fork by an overlay the deploy
  ships in the bundle (`vllm_fork.md`, "Kaggle"). Both targets run.
- *The benchmark.* Both targets run **seal sweep 3's 110**
  (`make_benchmark_seal_sweep3_110`): the 32 games of the mix the sweep reads
  its arms at — the official games but four trivial ones, and the re-arc-3
  games at their scaled baselines — as 110 entries at one pass, so a Kaggle run
  of it is what the sweep's box replay simulates. The **submission-shaped 110**
  (`make_benchmark_kaggle_official_110`, the 25 official games round-robin) is
  the one to swap in for a run shaped like the leaderboard's public set. The
  25-game set (`make_benchmark_official_25`) is too small to fill the presets'
  concurrency, so it would leave the engine under-fed. None is a competition
  run — `run_as_submission` stays off below.

All three come from `taaf.standard_benchmarks`, so the game set is
TAAF's definition rather than a list maintained here.

In [ ]:
from seal import SealAgent as Agent

agent = Agent.build(environment="slurm-b200" if TARGET == "slurm" else "kaggle-rtx")
solver = LLMAgentSolver(agent=agent)

# Seal sweep 3's 110 on BOTH targets: a 25-game set cannot fill the presets'
# concurrency, and this mix is what the sweep reads a Kaggle run of.
benchmark = taaf.standard_benchmarks.make_benchmark_seal_sweep3_110(solver=solver)
benchmark.label = f"demo_seal_{TARGET}"
benchmark.job_dir = JOB_DIR

print(
    f"{benchmark.label}: {len(benchmark.games)} games x {benchmark.n_passes} pass(es), "
    f"max_concurrent={agent.run_config.max_concurrent}"
)

## The deploy target

Both targets take the 30-minute budget through the same field,
`max_runtime_s`, and both measure it from **worker start** — engine load
included, not just play.

**SLURM** — one GPU on TufaSlurmTarget's default tier. The walltime is an
hour rather than the usual twelve: `max_runtime_s` is what actually stops
the run, and asking the scheduler for twelve hours to use thirty minutes
of them only costs queue priority. The resources are sized for the engine
rather than for the concurrency:

| | cpus/GPU | mem/GPU | why |
|---|---|---|---|
| seal | 24 | 260 GiB | sized for the heaviest column, Flash-Next on the B200: its engine, the pinned PLE table, the host KV tier the preset claims eagerly at start, and the game processes its concurrency runs |

`main_project` names the bundled repo the worker `uv sync`s from. It must
equal **this checkout's directory name**, so the shared deployment helper
detects it rather than relying on a constant that breaks in worktrees.

**Kaggle** — the shared helper uses TAAF's own resolver (`.env`, then
`KAGGLE_USERNAME`, then `~/.kaggle/kaggle.json`).
The kernel is the final solution's, and its bundle goes to the dataset the submitted
version reads rather than TAAF's default source dataset, one shared
`<owner>/taaf-release-source` that two agents deployed from one account
would push versions of over each other. The model snapshot and wheels sources still ride along on the
agent's `run_config`, forwarded by the solver, so there is nothing to
declare for those. `run_as_submission` stays off — a real submission
ignores the budget and runs to the competition's own limit.

In [ ]:
if TARGET == "slurm":
    # TAAF takes the EARLIER of the budget and the walltime minus a 10-minute
    # teardown buffer, so a budget that reaches the walltime is silently cut
    # short by it instead. Raise `time` first if you raise the budget.
    WALLTIME_S = 1 * 3600
    assert DEMO_BUDGET_S < WALLTIME_S - 600, "DEMO_BUDGET_S >= walltime - 10 min buffer — raise `time` too"

    # Both tiers include their CPU KV offload cache, which the engine
    # claims at start — the job has to ask for that memory up front. The
    # seal's tier is on by default so its concurrency can run past what
    # the pool holds (see seal.presets.configure). Sized for the heaviest
    # column, Flash-Next on the B200: its larger tier on top of the pinned
    # PLE table, and CPUs and memory for the game processes and kernels
    # its concurrency runs.
    cpus, mem_gib = 24, 260
    target = taaf.deploy_slurm.TufaSlurmTarget(
        time="01:00:00",
        max_runtime_s=DEMO_BUDGET_S,
        gpu_count=1,
        cpus_per_gpu=cpus,
        mem_per_gpu_bytes=mem_gib * 1024**3,
        main_project=platypus.deployment.detect_main_project(),
    )
else:
    owner = platypus.deployment.kaggle_owner()
    target = taaf.deploy_kaggle.KaggleTarget(
        username=owner,
        kernel_slug="tufa-labs-arc3-solution",
        kernel_title="Tufa Labs ARC3 solution",
        dataset_ref=f"{owner}/tufa-labs-arc3-solution-src",
        run_as_submission=False,
        # As the final submission ran: the games open first and the keepalive guards the
        # start-up. Without stealth TAAF refuses a startup deadline, so there is none.
        startup_stealth=False,
        # No deadline from machine boot (2026-10-29): every submission of the drafter branch
        # with one ended on it, about 55 minutes after it was submitted and from either region, on a
        # machine already up past it. Without one a submission plays, losing the time its machine
        # was up before the notebook started.
        max_runtime_s=DEMO_BUDGET_S,
    )
    # A submission's keepalive in a version run too, so its log shows the RESETs a submission sends;
    # at half TAAF's interval (2026-10-30).
    benchmark.keepalive_resets = True
    benchmark.keepalive_interval_s = 5 * 60

handle = await benchmark.deploy(target)
print("submitted:", getattr(handle, "job_id", None) or getattr(handle, "kernel_id", handle))
print("job_dir: ", handle.job_dir)

## Fire and forget

Nothing here waits. The job outlives this kernel, and re-running the
notebook always launches a **new** job into a new timestamped directory —
it never reattaches to a running one, so an accidental re-run costs an
allocation but can never disturb or overwrite a run in flight.

**The results are `diagnostics.html` in the job dir** — TAAF writes it
when the run finishes, and it holds the scores, the per-game breakdown
and the drill-downs.

**`make viewer` shows the run while it is still going.** It serves
`platypus.viewer` over the shared results tree (the same root the SLURM
job dir sits under, so a demo run shows up without configuring
anything), and its pages render from the diagnostic streams on every
request rather than from a finished file — so a run in flight is
readable, down to the LLM turn currently in progress. This is worth more
than the final page at a 30-minute budget: you can watch the first turn
and see whether the agent is doing anything sensible long before there
is a score. It reads over the filesystem, so a viewer on the login node
sees a run on any compute node.

For **SLURM** that is the job dir printed above, on shared storage,
alongside `stdout.log` / `stderr.log` and the per-game streams under
`games/`. For **Kaggle** neither applies: the run happens on Kaggle, so
there is nothing local to watch and the durable copy lives there — the
local job dir only ever held the uploaded bundle and it is in temp.
`handle.wait()` pulls the finished output down, and can pull it again
from the kernel id at any time.